# Dropout 为什么只在训练时启用？

**面试回答：**训练时随机屏蔽激活并按保留率缩放，推理时关闭随机性；否则同一请求会给不同结果。

## 真实案例

商品转化网络的 8 个隐藏单元用于预测活动点击。

In [1]:
import numpy as np  # 导入 NumPy 手写 dropout。
rng=np.random.default_rng(9)  # 固定随机数。
h=np.array([.2,.4,.6,.8,1.,1.2,1.4,1.6])  # 构造隐藏激活。
print('隐藏激活=',h.tolist())  # 输出真实样本激活。
print('单元数=',len(h))  # 输出规模。

隐藏激活= [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6]
单元数= 8


## Baseline / 基线

不使用 dropout 的推理输出固定为激活和。

In [2]:
baseline=float(h.sum())  # 计算无 dropout 的输出。
print('无dropout输出=',baseline)  # 输出基线。
print('基线不产生随机正则。')  # 说明作用。

无dropout输出= 7.2
基线不产生随机正则。


In [3]:
keep=.75  # 设置保留概率。
mask=(rng.random(len(h))<keep).astype(float)  # 训练时采样 Bernoulli 掩码。
train_output=float((h*mask/keep).sum())  # 用 inverted dropout 保持期望尺度。
inference_output=float(h.sum())  # 推理时关闭掩码。
wrong_inference=float((h*(rng.random(len(h))<keep)/keep).sum())  # 故意在推理继续随机屏蔽。
print('训练mask=',mask.astype(int).tolist(),'训练输出=',round(train_output,3))  # 输出训练中间量。
print('推理输出=',inference_output,'错误随机推理=',round(wrong_inference,3))  # 输出正确与错误推理。

训练mask= [0, 1, 1, 0, 1, 0, 0, 0] 训练输出= 2.667
推理输出= 7.2 错误随机推理= 4.0


## 结果解读

训练输出有随机性但期望接近基线；inverted 缩放使推理无需再乘 keep。dropout 不是缺失值填补。

In [4]:
samples=[]  # 保存多次训练输出。
for step in range(200):  # 重复采样验证期望。
    samples.append(float((h*(rng.random(len(h))<keep)/keep).sum()))  # 记录一次训练子网络输出。
print('训练输出均值=',round(float(np.mean(samples)),3),'基线=',baseline)  # 输出期望对齐证据。
print('生产差距：框架 train/eval 状态、随机种子和服务一致性必须测试。')  # 说明生产边界。

训练输出均值= 7.293 基线= 7.2
生产差距：框架 train/eval 状态、随机种子和服务一致性必须测试。


## 失败案例与修复

错误推理继续采样 mask 会令同一用户请求漂移；修复是 eval 时关闭 mask。

In [5]:
print('失败推理偏差=',round(abs(wrong_inference-baseline),3))  # 输出随机推理差异。
print('修复确定性输出=',inference_output)  # 输出关闭 dropout 的结果。
print('率过高会导致欠拟合，需要验证集选择。')  # 说明取舍。
print('不要把 MC Dropout 方差直接当作校准置信度。')  # 说明边界。

失败推理偏差= 3.2
修复确定性输出= 7.2
率过高会导致欠拟合，需要验证集选择。
不要把 MC Dropout 方差直接当作校准置信度。


In [6]:
assert len(h)>=5  # 保护单元数。
assert inference_output==baseline  # 保护推理关闭 dropout。
assert abs(np.mean(samples)-baseline)<.3  # 保护 inverted dropout 期望对齐。
assert wrong_inference!=baseline  # 保护失败案例。